In [0]:
from pyspark.sql import functions as F

silver_loads = spark.table(
    "workspace.transportation_analytics.silver_loads"
)

silver_routes = spark.table(
    "workspace.transportation_analytics.silver_routes"
)

In [0]:
silver_loads.printSchema()
silver_routes.printSchema()

In [0]:
monthly_loads = (
    silver_loads
    .withColumn(
        "month",
        F.date_trunc("month", "load_date")
    )
    .groupBy("month")
    .agg(
        F.countDistinct("load_id").alias("total_loads")
    )
    .orderBy("month")
)

display(monthly_loads)

In [0]:
monthly_revenue = (
    silver_loads
    .withColumn(
        "month",
        F.date_trunc("month", "load_date")
    )
    .groupBy("month")
    .agg(
        F.sum("revenue").alias("total_revenue")
    )
    .orderBy("month")
)

display(monthly_revenue)

In [0]:
freight_rate = (
    silver_loads
    .join(
        silver_routes.select(
            "route_id",
            "typical_distance_miles"
        ),
        on="route_id",
        how="left"
    )
    .withColumn(
        "month",
        F.date_trunc("month", "load_date")
    )
    .groupBy("month")
    .agg(
        F.round(
            F.sum("revenue") /
            F.sum("typical_distance_miles"),
            2
        ).alias("freight_rate_per_mile")
    )
    .orderBy("month")
)

display(freight_rate)

In [0]:
gold_seasonal_patterns = (
    monthly_loads
    .join(monthly_revenue, "month", "left")
    .join(freight_rate, "month", "left")
    .orderBy("month")
)

display(gold_seasonal_patterns)

In [0]:
gold_seasonal_patterns.limit(0).write \
    .format("delta") \
    .saveAsTable(
        "workspace.transportation_analytics.gold_seasonal_patterns"
    )

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.transportation_analytics.gold_seasonal_patterns"
)
target.alias("t").merge(
    gold_seasonal_patterns.alias("s"),
    "t.month = s.month"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.gold_seasonal_patterns"
    )
)

In [0]:
display(
    gold_seasonal_patterns.select(
        "month",
        "total_loads"
    ).orderBy("month")
)

Databricks visualization. Run in Databricks to view.

In [0]:
display(
    gold_seasonal_patterns.select(
        "month",
        "total_revenue"
    ).orderBy("month")
)

Databricks visualization. Run in Databricks to view.

In [0]:
display(
    gold_seasonal_patterns.select(
        "month",
        "freight_rate_per_mile"
    ).orderBy("month")
)

Databricks visualization. Run in Databricks to view.

In [0]:
gold_seasonal_patterns = gold_seasonal_patterns.withColumn(
    "season",
    F.when(F.month("month").isin(12, 1, 2), "Winter")
     .when(F.month("month").isin(3, 4, 5), "Spring")
     .when(F.month("month").isin(6, 7, 8), "Summer")
     .otherwise("Autumn")
)

display(gold_seasonal_patterns)

In [0]:
display(
    gold_seasonal_patterns
    .groupBy("season")
    .agg(
        F.sum("total_loads").alias("total_loads")
    )
    .orderBy("season")
)

Databricks visualization. Run in Databricks to view.

In [0]:
display(
    gold_seasonal_patterns
    .groupBy("season")
    .agg(
        F.avg("freight_rate_per_mile").alias(
            "avg_freight_rate_per_mile"
        )
    )
    .orderBy("season")
)

Databricks visualization. Run in Databricks to view.

In [0]:
spark.sql("""
ALTER TABLE workspace.transportation_analytics.gold_seasonal_patterns
ADD COLUMNS (season STRING)
""")

In [0]:
print(gold_seasonal_patterns.columns)

In [0]:
target.alias("t").merge(
    gold_seasonal_patterns.alias("s"),
    "t.month = s.month"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.gold_seasonal_patterns"
    ).select(
        "month",
        "season",
        "total_loads",
        "freight_rate_per_mile"
    ).orderBy("month")
)